In [87]:
import numpy as np
from dataclasses import dataclass, make_dataclass

from sklearn.datasets import load_iris, load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from geneticengine.grammar.grammar import extract_grammar
from geml.simplegp import SimpleGP

In [88]:
wine = load_wine()
X, y = wine.data, wine.target

In [89]:
n_features = X.shape[1]

fields = [(f'f{i}', bool) for i in range(n_features)]

FeatureMask = make_dataclass('FeatureMask', fields)

grammar = extract_grammar([FeatureMask], FeatureMask)

In [90]:
def fitness_function(mask: FeatureMask) -> float:
    selected_features = [getattr(mask, f'f{i}') for i in range(n_features)]
    if not any(selected_features):
        return 0.0 #useless
    X_subset = X[:, selected_features]

    clf = LogisticRegression(random_state=0, max_iter=200, solver='liblinear')
    f1 = cross_val_score(clf, X_subset, y, cv=3, scoring='f1_macro').mean()

    return f1

In [91]:
alg = SimpleGP(
    grammar=grammar,
    fitness_function=fitness_function,
    representation="treebased",
    minimize=False,
    seed=122,
    population_size=50,
    max_evaluations=10000,
    csv_output = "../gp_outputs/f1_scores.csv",
    # only_record_best_individuals=False, #default-True
)

In [92]:
best = alg.search()

In [93]:
best_individual = best[0]
best_f1 = best_individual.get_fitness(alg.get_problem())
print(best_f1)
print(best_individual.get_phenotype())

[0.9560467326267378]
FeatureMask(f0=True, f1=True, f2=True, f3=True, f4=False, f5=True, f6=True, f7=False, f8=False, f9=True, f10=False, f11=True, f12=True)
